# 04 — SHAP Explainability Analysis
**Project:** SARS-CoV-2 Genomic Signatures — Explainable Analysis  
**Author:** Ibrahim Mustafa (Bembo)

In [ ]:
!pip install shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle

print('Libraries loaded!')

In [ ]:
# Load model and data
with open('rf_model.pkl', 'rb') as f:
    rf = pickle.load(f)

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/your_file.csv', sep='\t')
df['Clade'] = df['Clade'].str.replace('.', '', regex=False).str.strip()
X = df.drop(columns=['ID', 'Clade'])
y = df['Clade']

sample_idx = df.sample(n=50000, random_state=42).index
X_rf = X.loc[sample_idx]
y_rf = y.loc[sample_idx]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42, stratify=y_rf)

X_shap = X_test.sample(n=500, random_state=42)
print(f'SHAP sample: {X_shap.shape}')

In [ ]:
# Calculate SHAP values
print('Calculating SHAP values...')
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)
print('Done!')

In [ ]:
# Global SHAP summary
shap.summary_plot(shap_values, X_shap,
                  class_names=rf.classes_,
                  plot_type='bar',
                  max_display=15,
                  show=False)
plt.title('SHAP Feature Importance per Clade')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP per clade — GRA, GK, GRY
class_names = list(rf.classes_)
gra_idx = class_names.index('Clade_GRA')
gk_idx  = class_names.index('Clade_GK')
gry_idx = class_names.index('Clade_GRY')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (idx, name) in zip(axes, [(gra_idx, 'GRA/Omicron'),
                                   (gk_idx,  'GK'),
                                   (gry_idx, 'GRY/Alpha')]):
    shap_df = pd.Series(
        abs(shap_values[idx]).mean(axis=0),
        index=X_shap.columns
    ).sort_values(ascending=False).head(10)

    shap_df.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top Features — Clade {name}')
    ax.set_xlabel('mean(|SHAP|)')
    ax.invert_yaxis()

plt.suptitle('Clade-Specific SHAP Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_per_clade.png', dpi=150)
plt.show()

print('Key Finding:')
print('GRA/Omicron: AT-rich motifs (ATA, TAT) dominate')
print('GK & GRY/Alpha: CpG-containing motifs (CGG, CCG) dominate')